# 06 — Feature Group Discovery


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from build_optimiser.config import Config
from build_optimiser.features import (
    expand_core,
    identify_core_libraries,
)
from build_optimiser.graph import attach_metrics, load_graph
from build_optimiser.partitioning import (
    bicluster_exe_library,
    extract_feature_groups,
    hierarchical_communities,
)

cfg = Config.from_yaml('../../config.yaml')
file_df = pd.read_parquet(cfg.processed_data_dir / 'file_metrics.parquet')
target_df = pd.read_parquet(cfg.processed_data_dir / 'target_metrics.parquet')
edge_df = pd.read_parquet(cfg.processed_data_dir / 'edge_list.parquet')
G = load_graph(cfg.processed_data_dir / 'edge_list.parquet')
attach_metrics(G, target_df)

results_dir = Path('../data/results')
results_dir.mkdir(parents=True, exist_ok=True)


## 1. Load Prerequisites

Read the executable × library matrix produced by notebook 05 and the target ownership produced by notebook 02.


In [ ]:
exe_lib_long = pd.read_parquet(cfg.processed_data_dir / 'exe_library_matrix.parquet')
ownership_df = pd.read_parquet(cfg.processed_data_dir / 'target_ownership.parquet')
groups_df = pd.read_parquet(cfg.processed_data_dir / 'contributor_groups.parquet')

# Derive owning_group_label from groups_df if not present
if 'owning_group_label' not in ownership_df.columns and 'owning_group_id' in ownership_df.columns:
    label_map = groups_df[['group_id', 'group_label']].drop_duplicates().set_index('group_id')['group_label']
    ownership_df['owning_group_label'] = ownership_df['owning_group_id'].map(label_map).fillna('unknown')

# Pivot long format to wide binary matrix (executables x libraries)
if 'executable' in exe_lib_long.columns:
    exe_lib_matrix = exe_lib_long.pivot_table(
        index='executable', columns='library', values='is_direct',
        aggfunc='max', fill_value=False,
    ).astype(int)
else:
    # Already in wide format
    exe_lib_matrix = exe_lib_long

print(f'Exe-library matrix: {exe_lib_matrix.shape[0]} executables x {exe_lib_matrix.shape[1]} libraries')
print(f'Ownership rows:     {len(ownership_df)}')
print()
print('Teams:', ownership_df['owning_group_label'].value_counts().to_dict())

## 2. Biclustering

Apply spectral co-clustering to the exe × library matrix after removing core libraries (those used by nearly all executables add noise rather than signal). Sweep K from 3 to 12 and plot within-cluster density to identify the elbow.


In [ ]:
# Remove core libraries before biclustering — they don't differentiate feature groups.
core_threshold = 0.50
core_libs = identify_core_libraries(exe_lib_long, threshold=core_threshold)
non_core_libs = [c for c in exe_lib_matrix.columns if c not in core_libs]

matrix_no_core = exe_lib_matrix[non_core_libs].copy()

print(f'Core libraries removed: {len(core_libs)}')
print(f'Non-core libraries for biclustering: {len(non_core_libs)}')
if matrix_no_core.size > 0:
    print(f'Matrix density after removal: {matrix_no_core.values.mean():.2%}')
else:
    print('No non-core libraries remaining.')

In [ ]:
# Sweep K from 3 to 12.
k_range = list(range(3, 13))

if matrix_no_core.shape[1] >= 3:
    bicluster_results = bicluster_exe_library(matrix_no_core, k_range=k_range)

    # Extract per-K scalar metrics from results list
    bc_df = pd.DataFrame([
        {'k': r['k'], 'within_density': r['within_density'], 'cross_fraction': r['cross_fraction']}
        for r in bicluster_results['results']
    ])
    print(f'Best K: {bicluster_results["best_k"]}')
    print(bc_df.to_string(index=False))
else:
    print(f'Only {matrix_no_core.shape[1]} non-core libraries — too few for biclustering, skipping.')
    bicluster_results = {'results': [], 'best_k': 1}
    bc_df = pd.DataFrame(columns=['k', 'within_density', 'cross_fraction'])

In [ ]:
if len(bc_df) >= 2:
    fig, ax = plt.subplots(figsize=(8, 4))

    density_col = next((c for c in bc_df.columns if 'density' in c.lower()), bc_df.columns[-1])
    k_col = next((c for c in bc_df.columns if 'k' in c.lower()), bc_df.columns[0])

    ax.plot(bc_df[k_col], bc_df[density_col], marker='o', color='steelblue')
    ax.set_xlabel('Number of biclusters (K)')
    ax.set_ylabel('Within-cluster density')
    ax.set_title('Biclustering: within-density vs K (look for elbow)')
    ax.set_xticks(bc_df[k_col].tolist())
    plt.tight_layout()
    plt.show()

    # Choose K at the elbow: largest second-derivative (sharpest bend).
    densities = bc_df[density_col].values.astype(float)
    if len(densities) >= 3:
        second_deriv = np.diff(np.diff(densities))
        best_k_idx = int(np.argmax(second_deriv)) + 1  # offset for double diff
        best_k = int(bc_df[k_col].iloc[best_k_idx])
    else:
        best_k = int(bc_df[k_col].iloc[len(bc_df) // 2])

    print(f'Suggested K (elbow): {best_k}')
else:
    best_k = bicluster_results.get('best_k', 1) or 1
    print(f'Insufficient data for elbow plot. Using best_k={best_k}.')

## 3. Hierarchical Community Detection

Apply Leiden / Louvain community detection with a range of resolution parameters to the full dependency graph. Plot modularity and community count vs resolution, and cross-group edges vs K, to find a partition that balances intra-community density with manageable group count.


In [ ]:
resolution_range = [0.2, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0]
community_results = hierarchical_communities(G, resolution_range=resolution_range)

# Extract per-resolution scalar metrics from results list
comm_df = pd.DataFrame([
    {
        'resolution': r['resolution'],
        'modularity': r['modularity'],
        'n_communities': r['n_communities'],
        'cross_community_edges': r['cross_community_edges'],
    }
    for r in community_results['results']
])

print(comm_df.to_string(index=False))

In [ ]:
res_col = next((c for c in comm_df.columns if 'res' in c.lower()), comm_df.columns[0])
mod_col = next((c for c in comm_df.columns if 'mod' in c.lower()), None)
count_col = next(
    (c for c in comm_df.columns if 'count' in c.lower() or 'communities' in c.lower() or 'k' == c.lower()),
    comm_df.columns[-1],
)
cross_col = next((c for c in comm_df.columns if 'cross' in c.lower() or 'cut' in c.lower()), None)

n_plots = 2 + (1 if cross_col else 0)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4))

ax_idx = 0
if mod_col:
    axes[ax_idx].plot(comm_df[res_col], comm_df[mod_col].astype(float), marker='o', color='steelblue')
    axes[ax_idx].set_xlabel('Resolution')
    axes[ax_idx].set_ylabel('Modularity')
    axes[ax_idx].set_title('Modularity vs resolution')
    ax_idx += 1

axes[ax_idx].plot(comm_df[res_col], comm_df[count_col].astype(float), marker='o', color='darkorange')
axes[ax_idx].set_xlabel('Resolution')
axes[ax_idx].set_ylabel('Community count')
axes[ax_idx].set_title('Community count vs resolution')
ax_idx += 1

if cross_col:
    axes[ax_idx].plot(comm_df[res_col], comm_df[cross_col].astype(float), marker='o', color='firebrick')
    axes[ax_idx].set_xlabel('Resolution')
    axes[ax_idx].set_ylabel('Cross-group edges')
    axes[ax_idx].set_title('Cross-group edges vs resolution')

plt.tight_layout()
plt.show()


## 4. Core Extraction

Identify core libraries (high reach) and expand the core to include their transitive dependencies. Report core size and compile time share.


In [ ]:
core_libs_set = identify_core_libraries(exe_lib_long, threshold=core_threshold)
expanded_core = expand_core(G, core_libs_set)

core_compile_ms = target_df[
    target_df['cmake_target'].isin(expanded_core)
]['compile_time_sum_ms'].sum()
total_compile_ms = target_df['compile_time_sum_ms'].sum()

print(f'Core libraries (reach >= {core_threshold:.0%}):    {len(core_libs_set)}')
print(f'Expanded core (+ transitive deps): {len(expanded_core)}')
print(f'Core compile time share:           {core_compile_ms / total_compile_ms:.1%}')
print()
print('Core targets (expanded):')
core_ct = (
    target_df[target_df['cmake_target'].isin(expanded_core)]
    [['cmake_target', 'target_type', 'compile_time_sum_ms']]
    .sort_values('compile_time_sum_ms', ascending=False)
)
print(core_ct.to_string(index=False))

## 5. Contributor Alignment

Overlay team ownership onto the proposed feature group partition. Compute NMI and ARI to quantify alignment between structural groups and organisational ownership. Identify targets where ownership and feature group membership disagree.


In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Pick a resolution for the feature groups. Use the first resolution that gives
# a community count close to best_k from biclustering.
target_k = best_k
comm_df_sorted = comm_df.copy()
comm_df_sorted['k_dist'] = (comm_df_sorted['n_communities'].astype(float) - target_k).abs()
best_resolution = float(comm_df_sorted.sort_values('k_dist').iloc[0]['resolution'])
print(f'Using resolution {best_resolution} (closest to K={target_k})')

# Get the partition at the chosen resolution from our earlier results.
# If the chosen resolution was in the original sweep, reuse it; otherwise re-run.
partition_at_res = None
for r in community_results['results']:
    if abs(r['resolution'] - best_resolution) < 1e-6:
        partition_at_res = r['partition']
        break

if partition_at_res is None:
    re_run = hierarchical_communities(G, resolution_range=[best_resolution])
    partition_at_res = re_run['results'][0]['partition']

# Build target -> feature_group_id mapping
feat_group_map = {t: int(gid) for t, gid in partition_at_res.items()}

print(f'Feature groups assigned to {len(feat_group_map)} targets')
print(f'Unique groups: {len(set(feat_group_map.values()))}')

In [ ]:
# Merge feature group assignments with ownership.
alignment_df = ownership_df[['cmake_target', 'owning_group_label', 'owning_group_id']].copy()
alignment_df['feature_group_id'] = alignment_df['cmake_target'].map(feat_group_map)
alignment_df = alignment_df.dropna(subset=['feature_group_id'])
alignment_df['feature_group_id'] = alignment_df['feature_group_id'].astype(int)

if len(alignment_df) > 1:
    nmi = normalized_mutual_info_score(
        alignment_df['owning_group_id'].astype(str),
        alignment_df['feature_group_id'].astype(str),
    )
    ari = adjusted_rand_score(
        alignment_df['owning_group_id'].astype(str),
        alignment_df['feature_group_id'].astype(str),
    )
    print(f'NMI (ownership vs feature groups): {nmi:.3f}')
    print(f'ARI (ownership vs feature groups): {ari:.3f}')
    print()
    if nmi < 0.5:
        print('Low NMI: structural feature groups are poorly aligned with team ownership.')
        print('Consider re-examining team boundaries or ownership assignments.')
    else:
        print('Good alignment between feature groups and team ownership.')
else:
    print('Not enough aligned targets to compute NMI/ARI')


In [ ]:
# Confusion matrix: feature group vs owning team.
if len(alignment_df) > 1:
    confusion = pd.crosstab(
        alignment_df['feature_group_id'],
        alignment_df['owning_group_label'],
        margins=False,
    )
    fig, ax = plt.subplots(figsize=(max(8, len(confusion.columns) * 1.2), max(6, len(confusion) * 0.8)))
    sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)
    ax.set_xlabel('Owning team')
    ax.set_ylabel('Feature group ID')
    ax.set_title('Feature group vs team ownership')
    plt.tight_layout()
    plt.show()

    # Identify misaligned targets: feature group is dominated by a different team than the owning team.
    group_dominant_team = (
        alignment_df.groupby('feature_group_id')['owning_group_label']
        .agg(lambda x: x.value_counts().index[0])
        .rename('dominant_team')
    )
    alignment_df = alignment_df.merge(group_dominant_team, on='feature_group_id')
    misaligned = alignment_df[alignment_df['owning_group_label'] != alignment_df['dominant_team']]
    print(f'\nMisaligned targets (owned by different team than group dominant): {len(misaligned)}')
    print(misaligned[['cmake_target', 'owning_group_label', 'feature_group_id', 'dominant_team']].head(20).to_string(index=False))


## 6. Feature Group Summary

For each proposed feature group, report the target count, executable count, compile time, SLOC, dominant team, cross-group dependency count, and self-containment score. Save the results.


In [ ]:
# Use extract_feature_groups to get the final feature group assignments.
# Pass the full community_results dict (which has 'results' key).
# Re-run at the chosen resolution if not already in our results.
communities_at_res = hierarchical_communities(G, resolution_range=[best_resolution])

feature_groups_df = extract_feature_groups(
    communities_at_res,
    core=expanded_core,
    ownership=ownership_df,
    resolution=best_resolution,
)

print(f'Feature group assignments: {len(feature_groups_df)} targets')
print(feature_groups_df['feature_group'].value_counts())

In [ ]:
# Build a summary DataFrame from the feature group assignments.
exe_set = set(exe_lib_matrix.index.tolist())
ownership_map = dict(zip(ownership_df['cmake_target'], ownership_df['owning_group_label']))
sloc_col = next((c for c in ['code_lines', 'sloc', 'code_lines_total'] if c in target_df.columns), None)

summary_rows = []
for gid, grp in feature_groups_df.groupby('feature_group'):
    members = set(grp['cmake_target'])
    member_df = target_df[target_df['cmake_target'].isin(members)]
    compile_time_s = member_df['compile_time_sum_ms'].sum() / 1000
    sloc = member_df[sloc_col].sum() if sloc_col else None
    exe_count = len(members & exe_set)

    # Dominant team
    teams = [ownership_map.get(t) for t in members if ownership_map.get(t)]
    dominant_team = pd.Series(teams).value_counts().index[0] if teams else 'unowned'

    # Cross-group edges
    cross_edges = sum(
        1
        for u, v in G.edges()
        if (u in members) != (v in members)
    )
    internal_edges = sum(
        1
        for u, v in G.edges()
        if u in members and v in members
    )
    total_incident = internal_edges + cross_edges
    self_containment = internal_edges / total_incident if total_incident > 0 else 1.0

    summary_rows.append({
        'feature_group': gid,
        'target_count': len(members),
        'exe_count': exe_count,
        'compile_time_s': compile_time_s,
        'sloc': sloc,
        'dominant_team': dominant_team,
        'cross_group_deps': cross_edges,
        'self_containment': self_containment,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('compile_time_s', ascending=False)
print(summary_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

group_label_col = 'feature_group' if 'feature_group' in summary_df.columns else 'label'

# Compile time by group.
axes[0].barh(summary_df[group_label_col].astype(str), summary_df['compile_time_s'], color='steelblue')
axes[0].set_xlabel('Compile time (s)')
axes[0].set_title('Compile time per feature group')
axes[0].invert_yaxis()

# Self-containment score.
colors = ['green' if s >= 0.7 else 'orange' if s >= 0.4 else 'red' for s in summary_df['self_containment']]
axes[1].barh(summary_df[group_label_col].astype(str), summary_df['self_containment'], color=colors)
axes[1].axvline(0.7, color='green', linestyle='--', alpha=0.5, label='Good (0.7)')
axes[1].axvline(0.4, color='orange', linestyle='--', alpha=0.5, label='Fair (0.4)')
axes[1].set_xlabel('Self-containment score')
axes[1].set_title('Group self-containment')
axes[1].set_xlim(0, 1)
axes[1].legend(fontsize=8)
axes[1].invert_yaxis()

# Cross-group dependencies.
axes[2].barh(summary_df[group_label_col].astype(str), summary_df['cross_group_deps'], color='firebrick')
axes[2].set_xlabel('Cross-group dependency edges')
axes[2].set_title('Cross-group coupling')
axes[2].invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# Save results.
results_dir = cfg.processed_data_dir.parent / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

summary_df.to_csv(results_dir / 'feature_groups.csv', index=False)
print(f'Saved feature_groups.csv ({len(summary_df)} groups)')

# Save per-target assignment parquet.
# feature_groups_df already has (cmake_target, feature_group) columns.
assignments_df = feature_groups_df.copy()
assignments_df.to_parquet(cfg.processed_data_dir / 'feature_group_assignments.parquet', index=False)
print(f'Saved feature_group_assignments.parquet ({len(assignments_df)} rows)')